# Conversational UI Chatbot App with ChatGPT, LangChain and Chainlit

Here we will build a advanced ChatGPT Conversational UI-based chatbot using LangChain and Chainlit with the following features:

- Custom Landing Page
- Conversational memory
- Result streaming capabilities (Real-time output)

## Install App and LLM dependencies

In [ ]:
!pip install langchain==0.3.11
!pip install langchain-openai==0.2.12
!pip install chainlit==1.3.2
!pip install pyngrok==7.2.2
!pip install pydantic==2.10.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.9/326.9 kB 20.0 MB/s eta 0:00:00
  Attempting uninstall: langsmith
    Found existing installation: langsmith 0.3.13
    Uninstalling langsmith-0.3.13:
      Successfully uninstalled langsmith-0.3.13
  Attempting uninstall: langchain
    Found existing installation: langchain 0.3.20
    Uninstalling langchain-0.3.20:
      Successfully uninstalled langchain-0.3.20
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 12.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.1/57.1 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of opentelemetry-semantic-conventions to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.7/169.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.3/455.3 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 44.0 MB/s eta 0:00:00
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.27.2
    Uninstalling pydantic_core-2.27.2:
      Successfully uninstalled pydantic_core-2.27.2
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.10.6
    Uninstalling pydantic-2.10.6:
      Successfully uninstalled pydantic-2.10.6


## Load OpenAI API Credentials

Here we load it from a file so we don't explore the credentials on the internet by mistake

In [ ]:
import locale
locale.getpreferredencoding = lambda: "UTF-8"

In [ ]:
import yaml

with open('api_keys.yml', 'r') as file:
    api_creds = yaml.safe_load(file)

In [ ]:
api_creds.keys()

dict_keys(['OPENAI_API_KEY', 'GEMINI_API_KEY', 'NGORK_AUTH_TOKEN', 'DEEPSEEK_API_KEY', 'TOGETHER_API_KEY', 'TAVILY_API_KEY'])

In [ ]:
import os

os.environ['OPENAI_API_KEY'] = api_creds['OPENAI_API_KEY']

## Write the app code here and store it in a py file

In [ ]:
%%writefile app.py
# the following line is a magic command
# that will write all the code below it to the python file app.py
# we will then deploy this app.py file on the cloud server where colab is running
# if you have your own server you can just write the code in app.py and deploy it directly


from langchain_openai import ChatOpenAI
from langchain.memory import ConversationBufferWindowMemory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain.schema.runnable.config import RunnableConfig
from langchain.schema import StrOutputParser
from operator import itemgetter
from dotenv import load_dotenv, find_dotenv
import chainlit as cl

@cl.on_chat_start
# this function is called when the app starts for the first time
async def when_chat_starts():

  # Load a connection to ChatGPT LLM
  load_dotenv('/home/santhosh/Projects/courses/Pinnacle/.env')
  chatgpt = ChatOpenAI(model_name='gpt-4o-mini', temperature=0.1,
                       streaming=True)

  # Add a basic system prompt for LLM behavior
  SYS_PROMPT = """
               Act as a helpful assistant and answer questions to the best of your ability.
               Do not make up answers.
               """

  # Create a prompt template for langchain to use history to answer user prompts
  prompt = ChatPromptTemplate.from_messages(
    [
      ("system", SYS_PROMPT),
      MessagesPlaceholder(variable_name="history"),
      ("human", "{input}"),
    ]
  )

  # Create a memory object to store conversation history window
  memory = ConversationBufferWindowMemory(k=20,
                                          return_messages=True)

  # Create a conversation chain
  conversation_chain = (
    RunnablePassthrough.assign(
      history=RunnableLambda(memory.load_memory_variables)
      |
      itemgetter("history")
    )
    |
    prompt
    |
    chatgpt
    |
    StrOutputParser() # to parse the output to show on UI
  )
  # Set session variables to be accessed when user enters prompts in the app
  cl.user_session.set("chain", conversation_chain)
  cl.user_session.set("memory", memory)


@cl.on_message
# this function is called whenever the user sends a prompt message in the app
async def on_user_message(message: cl.Message):

  # get the chain and memory objects from the session variables
  chain = cl.user_session.get("chain")
  memory = cl.user_session.get("memory")

  # this will store the response from ChatGPT LLM
  chatgpt_message = cl.Message(content="")

  # Stream the response from ChatGPT and show in real-time
  async for chunk in chain.astream(
    {"input": message.content},
    config=RunnableConfig(callbacks=[cl.LangchainCallbackHandler()]),
  ):
      await chatgpt_message.stream_token(chunk)
  # Finish displaying the full response from ChatGPT
  await chatgpt_message.send()
  # Store the current conversation in the memory object
  memory.save_context({"input": message.content},
                      {"output": chatgpt_message.content})

Writing app.py


## Start the app

In [ ]:
!chainlit run app.py --port=8989 --watch &>./logs.txt &

If the above command doesn't work, use the below command.

In [ ]:
!chainlit run app.py --port=8989

2025-03-07 11:11:56 - Created default chainlit markdown file at /content/chainlit.md
2025-03-07 11:11:56 - Your app is available at http://localhost:8989
E0000 00:00:1741345961.993603    8719 init.cc:232] grpc_wait_for_shutdown_with_timeout() timed out.


In [ ]:
from pyngrok import ngrok
import yaml

# Terminate open tunnels if exist
ngrok.kill()

# Setting the authtoken
# Get your authtoken from `ngrok_credentials.yml` file
# with open('ngrok_credentials.yml', 'r') as file:
#     NGROK_AUTH_TOKEN = yaml.safe_load(file)
ngrok.set_auth_token(api_creds['NGORK_AUTH_TOKEN'])

# Open an HTTPs tunnel on port XXXX which you get from your `logs.txt` file
ngrok_tunnel = ngrok.connect(8989)
print("Chainlit App:", ngrok_tunnel.public_url)

Chainlit App: https://0422-34-75-228-44.ngrok-free.app


## Change the Initial app screen

In [9]:
%%writefile chainlit.md

# Welcome I am AI Assistant 🤖

How can I help you today?

Overwriting chainlit.md


## Remove running app processes

In [10]:
ngrok.kill()

In [11]:
!ps -ef | grep app

root           7       1  1 05:03 ?        00:00:05 /tools/node/bin/node /datalab/web/app.js
root        1113       1  8 05:06 ?        00:00:06 /usr/bin/python3 /usr/local/bin/chainlit run app
root        1490     812  0 05:08 ?        00:00:00 /bin/bash -c ps -ef | grep app
root        1492    1490  0 05:08 ?        00:00:00 grep app


In [12]:
!sudo kill -9 1113